# Seminární práce 2

In [23]:
import wfdb
import numpy as np
from scipy import signal as sig
import matplotlib.pyplot as plt
import librosa as lib
with open("./voiced-database/RECORDS", "r") as f:
    records = f.readlines()
records = [record.strip() for record in records]

Pravděpodobně chybný záznam v anotacích, diagnóza je napsaná s překlepem a oficiálně je anotováno 58 zdravých hlasů, avšak po výpisu pouze zdravých jich naleznu pouze 57
voice143 ['<age>: 43 <sex>: F <diagnoses>: hyperkineti dysphonia <medications>: none']

In [100]:
def load_data(path, from_sample = 0,to_sample=None):
    signals, fields = wfdb.rdsamp(path, sampfrom=from_sample, sampto=to_sample)
    voice = signals[:, 0].flatten()
    return voice, fields

def bandpass_filter(signal, fs, lowcut=80, highcut=4000):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = sig.butter(4, [low, high], btype='band')
    return sig.filtfilt(b, a, signal)

def signal_cut(signal):
    ids = np.array(np.nonzero(signal))
    id1 = ids[0,0]
    #revert signal and calculate the last 
    SignalFlip = np.flip(signal)
    ids = np.array(np.nonzero(SignalFlip))
    id2 = signal.size - ids[0,0]-1
    SignalCut = signal[id1:id2]
    SignalCut = np.hamming(SignalCut.size)*SignalCut
    return SignalCut

def perform_fft(signal):
    signal = np.fft.fft(signal)
    signal = np.abs(signal)
    signal = np.log(signal)
    signal = np.fft.ifft(signal)
    signal = np.abs(signal)

    return np.array(signal)

def get_f0(signal, fs):
    minq = fs // 300
    maxq = fs // 70
    peaks = np.argmax(signal[minq:maxq]) + minq
    f0 = fs/peaks
    return f0

def classify_voice(f0, threshold=150):
    """ Klasifikace hlasu na základě F0 """
    return 1 if f0 > threshold else 0 # 0 healthy



In [93]:
i = 0
corr_healthy = 0
healthy = 0
pathol = 0
corr_healthy = 0
corr_pathol = 0
for record in records:
    if i == 210:
        break
    path = f"./voiced-database/{record}"
    voice, fields = load_data(path)
    fs = fields["fs"]
    #if "healthy" not in fields["comments"][0]:
    #if "hyperkinetic" in fields["comments"][0]:
    #         if "hypokinetic" not in fields["comments"][0]:
    #             if "reflux" not in fields["comments"][0]:
    #                 print(record, fields["comments"])
    voice = signal_cut(voice)
    voice = perform_fft(voice)
    f0 = get_f0(voice, fs)
    
    classif = classify_voice(f0)
    if classif == 0:
        healthy += 1
        if "heal" in fields["comments"][0]:
            corr_healthy += 1
    else:
        pathol += 1
        if "heal" not in fields["comments"][0]:
            corr_pathol += 1
    i += 1
print(healthy, corr_healthy)
print(pathol, corr_pathol)

57 13
151 107


In [ ]:
def mfcc(signal, fs):
    mfcc = lib.feature.mfcc(y= signal, sr = fs, n_mfcc=13)
    return np.mean(mfcc, axis=1)

def hnr(signal, sampling_rate=8000):
    stft = lib.stft(signal)
    magnitude = np.abs(stft)
    harmonic, noise = lib.decompose.hpss(magnitude)
    harmonic_energy = np.sum(harmonic**2, axis=0)
    noise_energy = np.sum(noise**2, axis=0)
    hnr = 10 * np.log10(harmonic_energy / (noise_energy + 1e-10))
    return np.mean(hnr)

def zcr(signal, sampling_rate=8000):
    zcr = lib.feature.zero_crossing_rate(signal)[0]
    return np.mean(zcr)

def spectral_centroid_(signal,fs):
    spectral_centroid = lib.feature.spectral_centroid(y=signal, sr=fs)
    return np.mean(spectral_centroid)

def classify_voice(hnr_val, mfcc_val, zcr_val, centroid_val):
    hnr_threshold = 22  # Zdravé hlasy mají vyšší HNR 21/22
    mfcc_threshold = -65  # Patologické mohou mít nižší hodnoty -65
    zcr_threshold = 0.35  # Vyšší ZCR = více šumu 0.35
    centroid_threshold = 970 # 970
    if hnr_val < hnr_threshold or zcr_val > zcr_threshold or centroid_val < centroid_threshold or mfcc_val < mfcc_threshold:
        return "Patologický"
    return "Zdravý"

In [193]:
corr_healthy = 0
healthy = 0
pathol = 0
corr_healthy = 0
corr_pathol = 0
for record in records[0:]:
    path = f"./voiced-database/{record}"
    voice, fields = load_data(path)
    fs = fields["fs"]
    voice = signal_cut(voice)
    mfccshit = np.mean(mfcc(voice, fs))
    hnrshit = hnr(voice, fs) 
    zcrshit = zcr(voice, fs)
    centroidshit = spectral_centroid_(voice, fs)
    classification = classify_voice(hnrshit, mfccshit, zcrshit,centroidshit)
    
    actual_class = "Zdravý" if "heal" in fields["comments"][0] else "Patologický"

    if classification == "Zdravý":
        healthy += 1
        if actual_class == "Zdravý":
            corr_healthy += 1
    else:
        pathol += 1
        if actual_class == "Patologický":
            corr_pathol += 1
print(healthy,corr_healthy, corr_healthy/58*100)
print(pathol, corr_pathol, corr_pathol/150*100)

56 25 43.103448275862064
152 120 80.0
